In [17]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [22]:
def shift_spectra_linear(
    spectra: torch.Tensor,
    wavegrid: torch.Tensor,
    velocities: torch.Tensor,
    extrapolate = "constant",
    return_mask: bool = False,
) -> torch.Tensor:
    """
        Doppler shift un ensemble de spectres en fonction d'une grille de longueurs d'onde et d'un ensemble de vitesses.

        ! Tous les spectres doivent être de manière idéalement contiguë pour de meilleures performances !
    Args:
        spectra (torch.Tensor): Batch de spectres à décaler, de forme [B, n_pixel].
        wavegrid (torch.Tensor): Batch de grilles de longueurs d'onde, de forme [B, n_pixel] (la même répétée).
        velocities (torch.Tensor): Vitesse de décalage Doppler, de forme [B, 1] ou [B].
        extrapolate (str): Méthode d'extrapolation à utiliser si la grille de longueurs d'onde est en dehors des limites des spectres.
            ('constant', 'zero', 'one', 'linear'). Par défaut, 'constant'.

    Returns:
        torch.Tensor: Spectres décalés, de forme [B, n_pixel].

    Example:
        >>> from dataset import SpectrumDataset
        >>> B = 32  # Batch size
        >>> dataset = SpectrumDataset(n_spectra=100, wavemin=5000, wavemax=5050, data_dtype=torch.float32)
        >>> batch_yobs = dataset.spectra
        >>> batch_wave = dataset.wavegrid.unsqueeze(0).expand(B, -1).contiguous()  # [B, n_pixel]
        >>> batch_voffset = torch.from_numpy(np.random.uniform(-3, 3, size=(B, 1))).cuda()
        >>> velocities = batch_voffset.view(-1, 1)
        >>> batch_yaug, extrap_mask = shift_spectra_linear(
        ...     spectra=batch_yobs,
        ...     wavegrid=batch_wave,
        ...     velocities=batch_voffset,
        ...     extrapolate="linear",
        ... )
    """
    # Constantes (handled in float64 downstream)

    # Vérifications de base
    assert spectra.shape == wavegrid.shape, (
        "Le spectre et la grille doivent avoir la même forme"
    )
    assert spectra.shape[0] == velocities.shape[0], (
        "Le nombre de spectres et le nombre de vitesses doivent correspondre"
    )
    assert extrapolate in ["constant", "zero", "one", "linear"], (
        "Extrapolation doit être l'une des valeurs suivantes : 'constant', 'zero', 'one', 'linear'"
    )

    # Ensure shapes and contiguity for best performance
    spectra = spectra.contiguous()
    wavegrid = wavegrid.contiguous()
    velocities = velocities.view(-1, 1).contiguous()

    # Compute Doppler factor with high precision to preserve tiny shifts
    # Use float64 math regardless of input dtypes, then cast back at the end
    c64 = torch.tensor(299_792_458.0, dtype=torch.float64, device=velocities.device)
    vel64 = velocities.to(torch.float64)
    doppler64 = torch.sqrt((1 + vel64 / c64) / (1 - vel64 / c64))

    # Shifted wavelength grid (float64 for accuracy)
    wave64 = wavegrid.to(torch.float64)
    shifted = (wave64 * doppler64).contiguous()

    # Interpolation via searchsorted
    idx = torch.searchsorted(shifted, wave64)

    # On récupère les indices pour les bords qui vont être extrapolés
    # idx == 0 signifie que la valeur de wavegrid est inférieure à la première valeur de shifted
    # idx == wavegrid.shape[-1] signifie que la valeur de wavegrid est supérieure à la dernière valeur de shifted
    mask_low = idx == 0
    mask_high = idx == wavegrid.shape[-1]
    extrap_mask = mask_low | mask_high

    idx = torch.clamp(idx, 1, spectra.shape[-1] - 1)

    # Extract intervals
    left_idx = idx - 1
    right_idx = idx

    λ_left = shifted.gather(-1, left_idx)
    λ_right = shifted.gather(-1, right_idx)
    # Work in float64 during interpolation, then cast back
    f_left = spectra.gather(-1, left_idx).to(torch.float64)
    f_right = spectra.gather(-1, right_idx).to(torch.float64)

    # Linear interpolation
    t = (wave64 - λ_left) / (λ_right - λ_left + 1e-12)
    result = f_left + t * (f_right - f_left)

    # Extrapolation si demandé
    if extrapolate == "zero":
        result = torch.where(extrap_mask, 0.0, result)
    elif extrapolate == "one":
        result = torch.where(extrap_mask, 1.0, result)
    elif extrapolate == "constant":
        # pour constant, on utilise f_left et f_right sur ce même mask
        result = torch.where(mask_low, f_left, result)
        result = torch.where(mask_high, f_right, result)

    # Cast result back to original dtype to preserve memory/perf characteristics
    result = result.to(spectra.dtype)

    if return_mask:
        return result, extrap_mask
    else:
        return result



In [ ]:
dset = np.load(
    "../data/soapgpu_ns3292_5000-5050_p100_k0p3_phi0_cubic.npz"
)

ccfs = np.load(
    ""
)

In [ ]:
wavegrid = dset["wavegrid"]
activity = dset["activity"]
template = dset["template"]

vgrid = np.arange(-20000, 20000, 250)  # m/s

vgrid_torch = torch.tensor(vgrid, dtype=torch.float32)

artificial_dataset = shift_spectra_linear(
    spectra=torch.tensor(template, dtype=torch.float32).unsqueeze(0).expand(len(vgrid), -1).contiguous(),
    wavegrid=torch.tensor(wavegrid, dtype=torch.float32).unsqueeze(0).expand(len(vgrid), -1).contiguous(),
    velocities=vgrid_torch.unsqueeze(1),
    extrapolate="constant",
)

ccf = np.load(
    
)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(soap_wavegrid, soap_spectra[0, :], label="Spectre", color="black")
plt.xlabel("Longueur d'onde (Å)")
plt.ylabel("Flux (normalisé)")
plt.xlim(5000, 5010)
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig(
    "/home/tliopis/Codes/exoplanets_llopis_mary_2025/notebooks/plots_rapport/contexte_scientifique/spectrum_example.png",
    dpi=300,
)
plt.show()